# Deposit Attrition EDA — v3

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v2 produced the first real result and four defects in how it was measured. v3
fixes the defects and adds the analysis that turns a curve into a capability.
It **reads v2's parquet panels and rebuilds nothing**, so it runs in minutes.

## What was wrong, and what changes

| | Defect | Fix |
|---|---|---|
| §7.1 | The lead detector searched from rel_m −12 — **inside its own baseline window**, where every feature equals 1.0 by construction. v2's 5-month headline came from `cpty_new_out` "separating" at −10, which is the window edge plus the discreteness of a 0–1 count | Search starts at **−9**; cells thinner than `MIN_CELL_N` are dropped; mechanical features excluded from the headline |
| §7.2 | Medians are not operating points. Attriters sit at 0.79 of baseline nine months out and stayers at 0.95 — nobody measured **how many stayers are also below 0.79** | §6 computes recall, FPR, precision at the observed monthly hazard, and alerts-per-true-positive across a threshold grid |
| §7.3 | `share_out_selfpay` was 0.000 for both cohorts at every rel_m because most customers never self-pay. The same-name-outflow hypothesis was **untested, not disproved** | §3 picks the statistic from the data; §7 tests self-pay three ways, including conditional on baseline self-payers |
| §7.4 | `cpty_out_hhi` runs 0.001 → 0.271, but fewer counterparties *arithmetically* means higher concentration | §8 holds the counterparty count fixed and asks whether HHI still separates |
| §7.5 | `mean_new_fi_out` was 12.4 in 2024-01 against ~0.5 later — in the first months every counterparty is new | New-entity features blanked for 2024-01…03 |

Also: `B_bal_exit` now gets the same event study as `A_full_exit` — the 3,063
customers who moved the money but kept the shell account open — and the labels
are persisted to parquet so v4 doesn't recompute them.

**The expected corrected headline is `amt_out` at rel_m −9 versus `bal_live` at
−5: a four-month lead, not five.** §5b prints both so the change is visible.

**§6 is the section that decides whether this ships.** Precision is computed at
the real monthly hazard (~0.9%), so it will be low. That is the base rate, not a
failure of the signal — and if no single feature clears the bar, that is the
argument for a model rather than a rule.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v3
# =====================================================================
# v3 fixes the four defects in the v2 event study and adds the analysis
# that turns a curve into a capability. It reads v2's parquet panels and
# rebuilds nothing, so it runs in minutes.
#
#   §7.1  the lead detector fired INSIDE its own baseline window
#   §7.2  medians are not operating points - precision was never measured
#   §7.3  sparse features are structurally zero at the median, so the
#         same-name-outflow hypothesis is untested, not disproved
#   §7.4  cpty_out_hhi rises mechanically as counterparties drop away
#   §7.5  new-entity features have a first-months burn-in
from pathlib import Path

DB           = "dsihd01p_dsi"
TBL_DEPOSITS = f"{DB}.lap_dsi_universe_optimized"

DATE_START   = "2024-01-01"
DATE_END     = "2026-07-31"

# v2 artefacts, read-only
HDFS_V2      = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_DIR     = "hdfs://nameservice1/user/pk36814/attrition_v3"
OUT_DIR      = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v3")

MAX_ROWS     = 60
ZERO_TOL     = 1.0

# ── Labels (unchanged from v2, recomputed and PERSISTED this time) ────
MIN_HIST_M      = 12
MIN_LIVE_BEFORE = 6
BAL_EXIT_FRAC   = 0.05
BAL_EXIT_HOLD   = 3

# ── Event study ───────────────────────────────────────────────────────
STUDY_DEFS   = ["A_full_exit", "B_bal_exit"]   # both, not just A
EVENT_PRE    = 12
EVENT_POST   = 3
BASE_WINDOW  = (-12, -10)

# FIX §7.1 — the separation search must start AFTER the baseline window.
# Every feature sits at ~1.0 inside it by construction, so a "separation"
# at rel_m -10 is an artefact of the window edge, not a signal. v2's
# 5-month headline rested entirely on that.
SEARCH_FROM  = BASE_WINDOW[1] + 1      # -9
MIN_CELL_N   = 200                     # ignore cells thinner than this
SEP_LEVEL    = 0.15
SEP_SHARE    = 0.03
SEP_RATE     = 0.05                    # for rate-of-any statistics
HOLD         = 2

# FIX §7.5 — in the first months every counterparty is new by
# construction (mean_new_fi_out was 12.4 in 2024-01 against ~0.5 later).
BURN_IN_YM   = ["2024-01", "2024-02", "2024-03"]
NEW_ENTITY   = ["cpty_new_out", "fin_new_out"]

# ── Operating points (FIX §7.2) ───────────────────────────────────────
# Ratio-to-own-baseline thresholds. A customer is flagged in month k if
# its feature has fallen below the threshold.
OP_THRESH    = [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]
OP_FEATURES  = ["amt_out", "amt_in", "n_out", "n_in", "net_flow",
                "bal_live", "cpty_out_n", "fin_out_n"]
TARGET_PREC  = 0.25       # what counts as a usable alert queue
MIN_RECALL   = 0.20

# ── Feature taxonomy — decides which statistic each feature gets ──────
# DENSE     ratio to own baseline, median          (§5.5 worked)
# SPARSE    rate-of-any across the cohort          (FIX §7.3)
# MECHANIC  reported, but never sold as independent (FIX §7.4)
DENSE = ["bal_live", "amt_out", "amt_in", "net_flow", "n_out", "n_in",
         "cpty_out_n", "fin_out_n"]
SPARSE = ["share_out_selfpay", "cpty_new_out", "fin_new_out",
          "share_out_check", "share_out_wire", "share_out_origpnc",
          "share_out_internal"]
MECHANIC = ["cpty_out_hhi", "cpty_out_top", "fin_out_hhi", "fin_out_top"]
SHARE_LIKE = ["share_out_ach", "share_out_card"]

In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS
# =====================================================================
import warnings, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

spark = (SparkSession.builder
         .appName("pkg_attrition_eda_v3")
         .config("spark.sql.shuffle.partitions", "400")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def hp(name):  return f"{HDFS_DIR.rstrip('/')}/{name}"
def v2(name):  return f"{HDFS_V2.rstrip('/')}/{name}"

_FIND = OUT_DIR / "FINDINGS_v3.csv"
FINDINGS = pd.read_csv(_FIND).to_dict("records") if _FIND.exists() else []

def note(qid, question, answer, detail=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=question, answer=str(answer), detail=str(detail)))
    pd.DataFrame(FINDINGS).to_csv(_FIND, index=False)

def _dec_safe(sdf):
    out = sdf
    for n, t in sdf.dtypes:
        if t.startswith("decimal"):
            out = out.withColumn(n, F.col(n).cast("double"))
    return out

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec_safe(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    return disp(pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())}),
                title=title, n=len(d), save=save)

def pct(a, b): return float(a) / float(b) if b else float("nan")

# ── load v2 panels ────────────────────────────────────────────────────
acct_month = spark.read.parquet(v2("panel_account_month")).persist(StorageLevel.DISK_ONLY)
cust_month = spark.read.parquet(v2("panel_customer_month")).persist(StorageLevel.DISK_ONLY)
feat       = spark.read.parquet(v2("panel_pay_features")).persist(StorageLevel.DISK_ONLY)

# FIX §7.5 — blank the new-entity counts during the burn-in rather than
# dropping the rows, so the other features keep their months.
for c in NEW_ENTITY:
    if c in feat.columns:
        feat = feat.withColumn(c, F.when(F.col("ym").isin(*BURN_IN_YM), None).otherwise(F.col(c)))

ALL_FEATS = [c for c in (DENSE + SPARSE + MECHANIC + SHARE_LIKE)
             if c in feat.columns or c == "bal_live"]

kv({"account-months": acct_month.count(),
    "customer-months (deposits)": cust_month.count(),
    "customer-months (payments)": feat.count(),
    "features carried": len(ALL_FEATS),
    "burn-in months blanked for new-entity features": ", ".join(BURN_IN_YM)},
   title="0 &middot; v2 panels loaded", save="v3_inputs")
print("features:", ALL_FEATS)

## 1 · Labels, rebuilt and persisted

In [ ]:
# =====================================================================
# 2 · LABELS, REBUILT AND PERSISTED                     [OUTPUT BLOCK 1]
# =====================================================================
# Identical logic to v2 §6, but written to parquet this time so nothing
# downstream has to recompute it.

w    = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
w3   = w.rangeBetween(-2, 0)
w12  = w.rangeBetween(-11, 0)
wfwd = w.rangeBetween(0, BAL_EXIT_HOLD - 1)

c = (cust_month
     .withColumn("n_hist", F.count("bal_live").over(w12))
     .withColumn("_a12", F.array_sort(F.collect_list("bal_live").over(w12)))
     .withColumn("med12", F.expr("element_at(_a12, cast(size(_a12)/2 as int) + 1)")).drop("_a12")
     .withColumn("low", ((F.col("med12") > ZERO_TOL) &
                         (F.col("bal_live") < BAL_EXIT_FRAC * F.col("med12"))).cast("int"))
     .withColumn("low_run", F.sum("low").over(wfwd))
     .withColumn("obs_fwd", F.count("*").over(wfwd))
     .withColumn("all_closed_run", F.sum("all_closed").over(wfwd)))

DEFS = {
 "A_full_exit": (F.col("all_closed") == 1) & (F.col("all_closed_run") == F.col("obs_fwd")),
 "B_bal_exit":  (F.col("n_hist") >= MIN_HIST_M) & (F.col("low_run") == F.lit(BAL_EXIT_HOLD)) &
                (F.col("obs_fwd") == BAL_EXIT_HOLD),
}
for k, cond in DEFS.items():
    c = c.withColumn(k, F.when(cond, 1).otherwise(0))

lab = (c.groupBy("cust_pwr_id").agg(
          *[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"m_{k}") for k in DEFS],
          F.min("m_idx").alias("first_m"), F.max("m_idx").alias("last_m"),
          F.count("*").alias("n_months"),
          F.min(F.when(F.col("n_accts_live") > 0, F.col("m_idx"))).alias("first_live_m"),
          F.sum((F.col("n_accts_live") > 0).cast("int")).alias("n_live_months"),
          F.max("segment_desc").alias("segment_desc"),
          F.expr("percentile_approx(bal_live, 0.5)").alias("median_bal"))
       .filter(f"n_months >= {MIN_HIST_M}"))

for k in DEFS:   # qualified: real tenure before the event
    lab = lab.withColumn(f"q_{k}", F.when(
        F.col(f"m_{k}").isNotNull() & F.col("first_live_m").isNotNull() &
        (F.col(f"m_{k}") - F.col("first_live_m") >= MIN_LIVE_BEFORE), F.col(f"m_{k}")))

lab.write.mode("overwrite").parquet(hp("labels_customer"))
lab = spark.read.parquet(hp("labels_customer")).persist(StorageLevel.DISK_ONLY)
N_EV = lab.count()

# ── prevalence, for the operating-point maths in §6 ───────────────────
_mspan = cust_month.agg((F.max("m_idx") - F.min("m_idx") + 1)).collect()[0][0]
EVAL_MONTHS = _mspan - MIN_HIST_M          # months in which an event can be observed
PREV = {}
rows = []
for k in DEFS:
    nq = lab.filter(F.col(f"q_{k}").isNotNull()).count()
    PREV[k] = pct(nq, N_EV * EVAL_MONTHS)
    rows.append(dict(definition=k,
                     n_events_raw=lab.filter(F.col(f"m_{k}").isNotNull()).count(),
                     n_events_qualified=nq,
                     share_of_customers=pct(nq, N_EV),
                     monthly_hazard=PREV[k],
                     full_pre_window=lab.filter(
                         F.col(f"q_{k}").isNotNull() &
                         (F.col(f"q_{k}") - F.col("first_m") >= EVENT_PRE)).count()))
disp(pd.DataFrame(rows), title=f"1a &middot; Labels (n={N_EV:,} evaluable customers, "
                               f"{EVAL_MONTHS} evaluable months)", save="v3_labels")

note("PREV", "Monthly hazard used for precision",
     "; ".join(f"{k}: {v:.4%}" for k, v in PREV.items()),
     "Events / (customers x evaluable months). This is the base rate an alerting queue "
     "actually faces, and it is what makes precision low even at high recall.")

## 2 · Sparsity audit — which statistic each feature deserves

In [ ]:
# =====================================================================
# 3 · SPARSITY AUDIT — WHICH STATISTIC EACH FEATURE DESERVES
#                                                       [OUTPUT BLOCK 2]
# =====================================================================
# FIX §7.3. v2 reported a median ratio-to-baseline for every feature. For
# share_out_selfpay that median was 0.000 for BOTH cohorts at every rel_m,
# with max_gap 0.000 - not because self-pay is uninformative but because
# most customers never self-pay, so the median is structurally zero.
# The same-name-outflow hypothesis was never tested.
#
# Decide the statistic from the data, not from the feature's name.

panel = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live", "n_accts", "segment_desc")
         .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left"))

_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in ALL_FEATS])
long_all = panel.select("cust_pwr_id", "m_idx",
                        F.expr(f"stack({len(ALL_FEATS)}, {_stack}) as (feature, value)"))

aud = (long_all.groupBy("feature").agg(
          F.count("*").alias("n_rows"),
          F.avg(F.col("value").isNotNull().cast("double")).alias("share_non_null"),
          F.avg(F.when(F.col("value").isNotNull(),
                       (F.abs(F.col("value")) > 1e-12).cast("double"))).alias("share_non_zero"),
          F.expr("percentile_approx(value, 0.5)").alias("median"),
          F.expr("percentile_approx(value, 0.9)").alias("p90"),
          F.countDistinct(F.when(F.col("value") > 0, F.col("cust_pwr_id"))).alias("n_cust_any"))
       .withColumn("share_cust_any", F.col("n_cust_any") / F.lit(N_EV)))

aud_p = aud.toPandas()
def _stat(r):
    if r.share_non_zero is None or r.share_non_zero < 0.25:
        return "RATE  (median is structurally zero)"
    if r.feature in MECHANIC:
        return "RATIO (mechanical - see 5b)"
    return "RATIO (ratio to own baseline)"
aud_p["statistic"] = aud_p.apply(_stat, axis=1)
aud_p = aud_p.sort_values("share_non_zero")
disp(aud_p, title="2a &middot; Sparsity audit — a feature non-zero on under a quarter of "
                  "customer-months cannot be read at the median", n=40, save="v3_sparsity")

RATE_FEATS  = sorted(set(aud_p.loc[aud_p.statistic.str.startswith("RATE"), "feature"]))
RATIO_FEATS = [f for f in ALL_FEATS if f not in RATE_FEATS]
print("RATE  statistic:", RATE_FEATS)
print("RATIO statistic:", RATIO_FEATS)

note("SPARSE", "Which features can be read at the median?",
     f"{len(RATIO_FEATS)} ratio-readable, {len(RATE_FEATS)} require a rate statistic",
     "v2 applied a median ratio to all 21 and concluded seven 'never separate'. "
     "For the sparse ones that was a property of the statistic, not the data.")

## 3 · Event study, both definitions, dual statistic

In [ ]:
# =====================================================================
# 4 · EVENT STUDY, BOTH DEFINITIONS, DUAL STATISTIC     [OUTPUT BLOCK 3]
# =====================================================================
# Ratio-readable features keep the v2 treatment (median ratio to the
# customer's own rel_m -12..-10 baseline). Sparse features get a RATE:
# the share of the cohort with any non-zero value that month. A rate needs
# no baseline, so it also sidesteps the division-by-zero that made the
# v2 sparse curves degenerate.

def cohorts_for(defn):
    ev_col = f"q_{defn}"
    attr = (lab.filter(F.col(ev_col).isNotNull())
            .select("cust_pwr_id", F.col(ev_col).alias("event_m"),
                    F.col("first_live_m").alias("first_m"), "last_m")
            .withColumn("cohort", F.lit("attriter")))
    draw = [r.event_m for r in attr.select("event_m").limit(2000).collect()]
    draw = draw[::max(1, len(draw)//300)][:300] or [0]
    arr  = F.array(*[F.lit(int(x)) for x in draw])
    ctrl = (lab.filter(F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNull())
            .withColumn("event_m", F.element_at(arr, (F.abs(F.hash("cust_pwr_id")) % len(draw)) + 1))
            .withColumn("first_m", F.coalesce("first_live_m", "first_m"))
            .select("cust_pwr_id", "event_m", "first_m", "last_m")
            .withColumn("cohort", F.lit("stayer")))
    return (attr.unionByName(ctrl)
            .filter((F.col("first_m") <= F.col("event_m") - EVENT_PRE) &
                    (F.col("last_m") >= F.col("event_m"))))

def normalise(defn):
    co = cohorts_for(defn)
    es = (panel.join(co, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-EVENT_PRE, EVENT_POST)))
    lg = es.select("cust_pwr_id", "cohort", "rel_m",
                   F.expr(f"stack({len(ALL_FEATS)}, {_stack}) as (feature, value)"))
    base = (lg.filter(F.col("rel_m").between(*BASE_WINDOW))
            .groupBy("cust_pwr_id", "feature").agg(F.avg("value").alias("base")))
    nm = (lg.join(base, ["cust_pwr_id", "feature"], "left")
          .withColumn("idx", F.when(F.abs(F.col("base")) > 1e-9, F.col("value") / F.col("base"))))
    return co, nm.persist(StorageLevel.DISK_ONLY)

CURVES, NORMS, COH = {}, {}, {}
for d in STUDY_DEFS:
    co, nm = normalise(d)
    COH[d], NORMS[d] = co, nm
    cur = (nm.groupBy("feature", "cohort", "rel_m").agg(
               F.count("*").alias("n"),
               F.sum(F.col("idx").isNotNull().cast("int")).alias("n_idx"),
               F.expr("percentile_approx(idx, 0.5)").alias("med_idx"),
               F.avg(F.when(F.col("value").isNotNull(),
                            (F.col("value") > 0).cast("double"))).alias("rate_any"),
               F.avg("value").alias("mean_value"))
           ).toPandas()
    # FIX: thin cells are noise, not signal
    cur.loc[cur.n < MIN_CELL_N, ["med_idx", "rate_any", "mean_value"]] = np.nan
    CURVES[d] = cur
    cur.to_csv(OUT_DIR / f"v3_curves_{d}.csv", index=False)
    kv({"definition": d,
        "attriters": COH[d].filter("cohort='attriter'").count(),
        "stayers": COH[d].filter("cohort='stayer'").count(),
        "customer-months": int(cur.n.sum() / max(len(ALL_FEATS), 1))},
       title=f"3a &middot; Cohorts — {d}", save=f"v3_cohorts_{d}")

def curve_table(defn, feats, col):
    cur = CURVES[defn]
    p = (cur[cur.feature.isin(feats)]
         .pivot_table(index="rel_m", columns=["feature", "cohort"], values=col)
         .reindex(columns=pd.MultiIndex.from_product([feats, ["attriter", "stayer"]]))
         .round(3))
    return p.reset_index()

HEAD = [f for f in ["bal_live", "amt_out", "amt_in", "n_out", "cpty_out_n"] if f in RATIO_FEATS]
disp(curve_table("A_full_exit", HEAD, "med_idx"),
     title="3b &middot; A_full_exit — median ratio to own baseline (dense features)",
     n=EVENT_PRE + EVENT_POST + 1, save="v3_curve_A_dense")
disp(curve_table("B_bal_exit", HEAD, "med_idx"),
     title="3c &middot; B_bal_exit — the shell-account population, same features",
     n=EVENT_PRE + EVENT_POST + 1, save="v3_curve_B_dense")

RH = [f for f in RATE_FEATS if f in ("share_out_selfpay", "fin_new_out", "cpty_new_out")]
if RH:
    disp(curve_table("A_full_exit", RH, "rate_any"),
         title="3d &middot; A_full_exit — RATE of any non-zero value (the statistic v2 lacked)",
         n=EVENT_PRE + EVENT_POST + 1, save="v3_curve_A_rate")

## 4 · Lead detector, fixed

In [ ]:
# =====================================================================
# 5 · LEAD DETECTOR, FIXED                              [OUTPUT BLOCK 4]
# =====================================================================
# FIX §7.1. Three changes from v2:
#   - the search starts at SEARCH_FROM (-9), outside the baseline window.
#     v2 searched from -12, where every feature equals its own baseline by
#     construction, and its 5-month headline came from cpty_new_out
#     "separating" at -10. That was the window edge, not a signal.
#   - cells thinner than MIN_CELL_N are dropped, not read.
#   - rate features are compared on the rate, with their own threshold.

def separations(defn):
    cur = CURVES[defn]
    rows = []
    for f, g in cur.groupby("feature"):
        if f in RATE_FEATS:
            col, thr, kind = "rate_any", SEP_RATE, "rate"
        elif f in MECHANIC or f in SHARE_LIKE:
            col, thr, kind = "med_idx", SEP_SHARE, "share"
        else:
            col, thr, kind = "med_idx", SEP_LEVEL, "level"
        w = (g.pivot_table(index="rel_m", columns="cohort", values=col)
               .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if w.empty:
            continue
        gap = (w.attriter - w.stayer).abs()
        search = gap[gap.index >= SEARCH_FROM]          # <- the fix
        sep, run = None, 0
        for rm, v in search.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD:
                sep = rm - HOLD + 1
                break
        rows.append(dict(feature=f, kind=kind, threshold=thr,
                         first_separation_rel_m=sep,
                         lead_months=(None if sep is None else -sep),
                         gap_at_minus6=round(gap.get(-6, np.nan), 3),
                         gap_at_event=round(gap.get(0, np.nan), 3),
                         max_gap_in_search=round(search.max(), 3) if len(search) else np.nan,
                         mechanical=f in MECHANIC))
    return pd.DataFrame(rows).sort_values(
        ["first_separation_rel_m", "max_gap_in_search"], ascending=[True, False], na_position="last")

for d in STUDY_DEFS:
    t = separations(d)
    disp(t, title=f"4a &middot; First sustained separation, search restricted to rel_m &ge; "
                  f"{SEARCH_FROM} — {d}", n=40, save=f"v3_lead_{d}")
    if d == "A_full_exit":
        LEAD_A = t

_bal = LEAD_A[LEAD_A.feature == "bal_live"]
_pay = LEAD_A[(LEAD_A.feature != "bal_live") & LEAD_A.first_separation_rel_m.notna() &
              (~LEAD_A.mechanical)]
bal_m = None if _bal.empty else _bal.iloc[0].first_separation_rel_m
pay   = None if _pay.empty else _pay.iloc[0]

kv({"balance (bal_live) separates at rel_m": bal_m,
    "earliest non-mechanical payment feature": None if pay is None else pay.feature,
    "  ...separates at rel_m": None if pay is None else pay.first_separation_rel_m,
    "PAYMENT LEAD OVER BALANCE (months)":
        None if (pay is None or bal_m is None) else int(bal_m - pay.first_separation_rel_m),
    "v2 printed (wrong)": 5},
   title="4b &middot; Corrected headline", save="v3_headline")

note("LEAD", "Does payment behaviour lead the deposit balance?",
     ("INCONCLUSIVE" if (pay is None or bal_m is None) else
      f"YES - {pay.feature} at rel_m {int(pay.first_separation_rel_m)} vs bal_live at "
      f"{int(bal_m)}: {int(bal_m - pay.first_separation_rel_m)} months"),
     "v2 printed 5 months on the strength of cpty_new_out separating at rel_m -10, which sits "
     "at the edge of the baseline window where every feature is ~1.0 by construction. The "
     "search now starts at -9. Mechanical features are excluded from the headline.")

## 5 · Operating points — the missing half

In [ ]:
# =====================================================================
# 6 · OPERATING POINTS — THE MISSING HALF               [OUTPUT BLOCK 5]
# =====================================================================
# FIX §7.2. The event curves say the attriter MEDIAN is at 0.79 of baseline
# nine months out while the stayer median sits at 0.95. They do not say how
# many stayers are also below 0.79 - and that is the whole question for an
# alert queue.
#
# Recall and FPR are prevalence-free. Precision is not, so it is computed
# at the observed monthly hazard from §2 and reported alongside "alerts per
# true positive", which is the number a sales lead will actually ask for.

def op_points(defn):
    nm = NORMS[defn]
    agg = [F.count("*").alias("n"),
           F.sum(F.col("idx").isNotNull().cast("int")).alias("n_idx")]
    agg += [F.sum((F.col("idx") < t).cast("int")).alias(f"lt{i}")
            for i, t in enumerate(OP_THRESH)]
    raw = (nm.filter(F.col("feature").isin(*OP_FEATURES) &
                     F.col("rel_m").between(-EVENT_PRE, -1))
             .groupBy("feature", "rel_m", "cohort").agg(*agg)).toPandas()

    p = PREV[defn]
    out = []
    for (f, rm), g in raw.groupby(["feature", "rel_m"]):
        a = g[g.cohort == "attriter"]
        s = g[g.cohort == "stayer"]
        if a.empty or s.empty or a.iloc[0].n_idx < MIN_CELL_N or s.iloc[0].n_idx < MIN_CELL_N:
            continue
        a, s = a.iloc[0], s.iloc[0]
        for i, t in enumerate(OP_THRESH):
            rec = a[f"lt{i}"] / a.n_idx
            fpr = s[f"lt{i}"] / s.n_idx
            den = p * rec + (1 - p) * fpr
            prec = (p * rec / den) if den > 0 else np.nan
            out.append(dict(feature=f, rel_m=int(rm), threshold=t,
                            recall=rec, fpr=fpr, precision=prec,
                            alerts_per_true_positive=(1 / prec if prec and prec > 0 else np.nan),
                            n_attriter=int(a.n_idx), n_stayer=int(s.n_idx)))
    return pd.DataFrame(out)

OP = {}
for d in STUDY_DEFS:
    OP[d] = op_points(d)
    OP[d].to_csv(OUT_DIR / f"v3_operating_points_{d}.csv", index=False)

# ── headline: best usable operating point per feature per month ───────
def best(df, target=TARGET_PREC, min_rec=MIN_RECALL):
    ok = df[(df.precision >= target) & (df.recall >= min_rec)]
    if ok.empty:
        # nothing reaches the target - report the best precision available
        idx = df.groupby(["feature", "rel_m"]).precision.idxmax().dropna()
        return df.loc[idx].assign(meets_target=False)
    idx = ok.groupby(["feature", "rel_m"]).recall.idxmax()
    return ok.loc[idx].assign(meets_target=True)

b = best(OP["A_full_exit"])
piv = (b.pivot_table(index="feature", columns="rel_m", values="recall").round(3))
disp(piv.reset_index(), title=f"5a &middot; Recall at precision &ge; {TARGET_PREC:.0%} "
                              f"(A_full_exit). Blank = no threshold reaches it that month",
     n=20, save="v3_op_recall_by_month")

pp = (b.pivot_table(index="feature", columns="rel_m", values="precision").round(3))
disp(pp.reset_index(), title="5b &middot; Best achievable precision by month before exit",
     n=20, save="v3_op_precision_by_month")

# ── the single table to brief from ────────────────────────────────────
brief = []
for f, g in OP["A_full_exit"].groupby("feature"):
    ok = g[(g.precision >= TARGET_PREC) & (g.recall >= MIN_RECALL)]
    if ok.empty:
        brief.append(dict(feature=f, earliest_usable_rel_m=None, threshold=None,
                          recall=None, precision=None, alerts_per_tp=None,
                          best_precision_anywhere=round(g.precision.max(), 3)))
        continue
    e = ok.loc[ok.rel_m.idxmin()]
    brief.append(dict(feature=f, earliest_usable_rel_m=int(e.rel_m), threshold=e.threshold,
                      recall=round(e.recall, 3), precision=round(e.precision, 3),
                      alerts_per_tp=round(e.alerts_per_true_positive, 1),
                      best_precision_anywhere=round(g.precision.max(), 3)))
brief = pd.DataFrame(brief).sort_values("earliest_usable_rel_m", na_position="last")
disp(brief, title=f"5c &middot; Earliest month a feature supports a usable queue "
                  f"(precision &ge; {TARGET_PREC:.0%}, recall &ge; {MIN_RECALL:.0%})",
     n=20, save="v3_op_brief")

_u = brief[brief.earliest_usable_rel_m.notna()]
note("OPPOINT", "Can the lead be operationalised?",
     ("NO - no single feature reaches the precision target at any month" if _u.empty else
      f"{_u.iloc[0].feature} at rel_m {int(_u.iloc[0].earliest_usable_rel_m)}: "
      f"recall {_u.iloc[0].recall:.0%}, precision {_u.iloc[0].precision:.0%}, "
      f"{_u.iloc[0].alerts_per_tp:.0f} alerts per true positive"),
     f"Monthly hazard {PREV['A_full_exit']:.3%}. A low precision here is not a failure of the "
     "signal - it is the base rate. If no single feature clears the bar, that is the argument "
     "for a model over a rule, and 5b shows how much headroom there is.")

## 6 · Same-name outflow, tested properly

In [ ]:
# =====================================================================
# 7 · SAME-NAME OUTFLOW, TESTED PROPERLY                [OUTPUT BLOCK 6]
# =====================================================================
# FIX §7.3. Money leaving to an account in the customer's OWN name at
# another institution is the cleanest "moving to a competitor" evidence in
# this data. v2 reported a median that was 0.000 for both cohorts at every
# rel_m and concluded nothing. Three honest tests instead:
#
#   1. How many customers ever do it at all?
#   2. Does the RATE of doing it rise before exit?
#   3. Among customers who already do it, does the SHARE of outflow rise?
#
# Test 3 is the one that matters: it removes the population that never
# self-pays, which is what made the median useless.

SP = "share_out_selfpay"
nm = NORMS["A_full_exit"]

# 1 ── population
pop = (feat.groupBy("cust_pwr_id")
       .agg(F.max(F.when(F.col(SP) > 0, 1).otherwise(0)).alias("ever_selfpay"),
            F.avg(SP).alias("mean_share"))
       .join(lab.select("cust_pwr_id", "q_A_full_exit"), "cust_pwr_id", "inner")
       .withColumn("cohort", F.when(F.col("q_A_full_exit").isNotNull(), "attriter").otherwise("stayer")))
disp(pop.groupBy("cohort").agg(F.count("*").alias("n_customers"),
                               F.avg("ever_selfpay").alias("share_ever_selfpay"),
                               F.avg("mean_share").alias("mean_selfpay_share")),
     title="6a &middot; Who ever pays themselves at another bank?", save="v3_selfpay_population")

# 2 ── rate of any self-pay, by month before exit
r2 = (nm.filter(F.col("feature") == SP)
      .groupBy("cohort", "rel_m").agg(
          F.count("*").alias("n"),
          F.avg(F.when(F.col("value").isNotNull(), (F.col("value") > 0).cast("double"))).alias("rate_any"),
          F.avg("value").alias("mean_share"),
          F.expr("percentile_approx(value, 0.9)").alias("p90_share")))
disp(r2.orderBy("rel_m", "cohort"),
     title="6b &middot; RATE of any same-name outflow — the statistic v2 was missing",
     n=40, save="v3_selfpay_rate")

# 3 ── conditional on already self-paying in the baseline window
basers = (nm.filter((F.col("feature") == SP) & F.col("rel_m").between(*BASE_WINDOW))
          .groupBy("cust_pwr_id", "cohort").agg(F.max("value").alias("base_max"))
          .filter("base_max > 0").select("cust_pwr_id"))
r3 = (nm.filter(F.col("feature") == SP).join(basers, "cust_pwr_id", "inner")
      .groupBy("cohort", "rel_m").agg(
          F.count("*").alias("n"),
          F.expr("percentile_approx(value, 0.5)").alias("median_share"),
          F.avg("value").alias("mean_share"),
          F.expr("percentile_approx(idx, 0.5)").alias("median_ratio_to_baseline")))
disp(r3.orderBy("rel_m", "cohort"),
     title="6c &middot; Conditional on self-paying at baseline — does the share RISE?",
     n=40, save="v3_selfpay_conditional")

_p = r2.toPandas().pivot_table(index="rel_m", columns="cohort", values="rate_any")
_gap = (_p.get("attriter") - _p.get("stayer")) if {"attriter", "stayer"} <= set(_p.columns) else None
note("SELFPAY", "Does same-name outflow rise before exit?",
     ("no comparable cells" if _gap is None else
      f"max rate gap {_gap.abs().max():.3f} at rel_m {int(_gap.abs().idxmax())}"),
     "Three tests in 6a-6c. 6c is the decisive one: it drops customers who never self-pay, "
     "which is the population that made the v2 median structurally zero. A rise in the "
     "conditional share is direct evidence of money moving to a competitor rather than "
     "the business simply contracting.")

## 7 · Is HHI anything beyond counterparty count? · and findings

In [ ]:
# =====================================================================
# 8 · IS HHI ANYTHING BEYOND COUNTERPARTY COUNT?        [OUTPUT BLOCK 7]
# =====================================================================
# FIX §7.4. cpty_out_hhi runs 0.001 -> 0.271 across the attriter window.
# Fewer counterparties mechanically implies higher concentration, so the
# rise may carry no information beyond cpty_out_n. Test it by holding the
# count fixed: bucket on cpty_out_n and compare HHI within bucket. If the
# attriter/stayer gap disappears inside buckets, the feature is mechanical
# and must not be briefed as independent evidence.

es = (panel.join(COH["A_full_exit"], "cust_pwr_id", "inner")
      .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
      .filter(F.col("rel_m").between(-EVENT_PRE, 0))
      .filter(F.col("cpty_out_n").isNotNull() & F.col("cpty_out_hhi").isNotNull()))

buck = (es.withColumn("n_bucket",
                      F.when(F.col("cpty_out_n") <= 2, "1-2")
                       .when(F.col("cpty_out_n") <= 5, "3-5")
                       .when(F.col("cpty_out_n") <= 15, "6-15")
                       .when(F.col("cpty_out_n") <= 50, "16-50")
                       .otherwise("51+")))

uncond = (buck.groupBy("cohort").agg(
              F.expr("percentile_approx(cpty_out_hhi, 0.5)").alias("median_hhi"),
              F.expr("percentile_approx(cpty_out_n, 0.5)").alias("median_n"),
              F.count("*").alias("n")))
disp(uncond, title="7a &middot; Unconditional — HHI and counterparty count move together",
     save="v3_hhi_uncond")

cond = (buck.groupBy("n_bucket", "cohort").agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(cpty_out_hhi, 0.5)").alias("median_hhi"))
        .filter(F.col("n") >= MIN_CELL_N))
cp = (cond.toPandas().pivot_table(index="n_bucket", columns="cohort", values="median_hhi")
      .reindex(["1-2", "3-5", "6-15", "16-50", "51+"]))
cp["gap"] = (cp.get("attriter", np.nan) - cp.get("stayer", np.nan)).abs().round(4)
disp(cp.reset_index(), title="7b &middot; Within a fixed counterparty-count bucket, does HHI "
                             "still separate? A small gap means the feature is mechanical",
     save="v3_hhi_conditional")

_maxgap = cp["gap"].max() if "gap" in cp else np.nan
note("HHI", "Is cpty_out_hhi independent of cpty_out_n?",
     f"max within-bucket median gap {_maxgap:.4f}" if pd.notna(_maxgap) else "insufficient cells",
     "A gap near zero means the HHI rise is arithmetic - fewer counterparties, higher "
     "concentration - and it must not be presented as a second signal. Same reasoning "
     "applies to cpty_out_top, fin_out_hhi and fin_out_top.")

# =====================================================================
# 9 · FINDINGS                                          [OUTPUT BLOCK 8]
# =====================================================================
reg = pd.DataFrame(FINDINGS)
order = ["PREV", "SPARSE", "LEAD", "OPPOINT", "SELFPAY", "HHI"]
reg["_o"] = reg["id"].apply(lambda x: order.index(x) if x in order else 99)
disp(reg.sort_values("_o").drop(columns="_o"),
     title="8 &middot; v3 findings", n=40, save="FINDINGS_v3")

print("\nLocal:", OUT_DIR)
for f in sorted(OUT_DIR.glob("*.csv")):
    print("  ", f.name)
print("\nHDFS:", HDFS_DIR, "-> labels_customer")

---

## After this run

1. **Read §5b before anything else.** If the corrected lead is 4 months on
   `amt_out`, that is the number to brief — not v2's 5.
2. **§5c is the go/no-go.** If some feature supports precision ≥ 25% at recall
   ≥ 20% six months out, there is a shippable rule today. If nothing clears it,
   the finding is real but needs a model to exploit, and §5b shows the headroom.
3. **§6c decides the competitor-vs-contraction question.** A rising conditional
   self-pay share is direct evidence of money moving rather than a business
   shrinking. A flat one means this data cannot separate the two and the brief's
   second question needs different evidence.
4. **§7 governs what goes on a slide.** If the within-bucket HHI gap is near
   zero, drop `cpty_out_hhi`, `cpty_out_top`, `fin_out_hhi` and `fin_out_top`
   from the narrative — they are arithmetic, not behaviour.
5. **Then re-pull from 2023-01.** Only 12,138 of 16,384 qualified attriters have
   a full pre-window; a 2023 start recovers roughly a third of the cohort.
6. **Then model.** Discrete-time hazard, rolling-origin split on `m_idx`,
   features from `panel_pay_features`, nothing at or after t+1 in the feature
   set. 158,066 censored accounts make hazard the right frame over fixed-window
   classification.